# 00 · Build the corpus

Ingests every source of the experiment's corpus version, normalizes, groups into discourse
units, applies QC, and writes a hash-identified corpus artifact.

The hash printed at the end identifies this corpus everywhere downstream: split
directories are named after it, and every model and result records it. Re-running this
notebook on the same sources reproduces the same hash.

In [ ]:
EXPERIMENT = "baseline"

In [ ]:
from qomlaq import paths
from qomlaq.corpus import build_corpus, largest_groups, save_corpus
from qomlaq.experiments import load_experiment

exp = load_experiment(EXPERIMENT)
corpus = build_corpus(paths.data_dir(), exp.corpus)
save_corpus(corpus, paths.artifacts_dir())
print(f"{corpus.name}: {corpus.sha256}")

## What survived QC

Counts are post-filtering. Title pairs (chapter/section/fragment names) are reported
separately from lines, because they are corpus rows but not running text.

In [ ]:
corpus.summary()

In [ ]:
import pandas as pd
pd.DataFrame(corpus.manifest["qc_ledger"]).drop(columns="removed_by_source")

## Discourse units

Every row belongs to a unit that the splitter keeps intact. The largest units bound how
evenly the corpus can be split — a 330-row chapter cannot be divided.

In [ ]:
largest_groups(corpus, 10)

In [ ]:
from qomlaq.checks import check_no_line_groups, check_titles_have_content

for result in (check_no_line_groups(corpus.frame), check_titles_have_content(corpus.frame)):
    print(result)

## Next

`01_make_splits.ipynb` builds the experiment's splits from this corpus.